In [8]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv
import os

In [9]:
load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

model = ChatOpenAI(
    model="openai/gpt-oss-120b",
    base_url="https://api.groq.com/openai/v1",
    api_key=api_key
)

In [10]:
class blogstate(TypedDict):
    title : str
    outline: str
    content : str

In [11]:
def create_outline(state : blogstate) -> blogstate:

    #fetch title
    title = state["title"]

    #call llm gen outline
    prompt = f'generate a summary outline for a topic - {title}'

    state["outline"]= model.invoke(prompt).content

    return state

In [12]:
def create_blog(state : blogstate) -> blogstate:
    #fetch title
    title = state["title"]

    outline = state["outline"]

    prompt = f'generate a summary blog for a title - {title} using the given outline- {outline}'

    state["content"]= model.invoke(prompt).content

    return state


In [13]:
graph = StateGraph(blogstate)

#nodes
graph.add_node("create_outline", create_outline)
graph.add_node("create_blog", create_blog)

#edge
graph.add_edge(START, "create_outline")
graph.add_edge("create_blog", END)

#compile
workflow = graph.compile()

In [14]:

#execute the graph
initial_state = {"title": "bangladesh"}

final_state = workflow.invoke(initial_state)

print(final_state)

{'title': 'bangladesh', 'outline': '**Bangladesh – Summary Outline**\n\n---\n\n### I. Introduction  \n- **Location:** South‑Asia, bordered by India (west, north, east), Myanmar (southeast) and the Bay of Bengal (south).  \n- **Capital & Largest City:** Dhaka.  \n- **Population:** ~170\u202fmillion (2024) – 8th‑largest in the world.  \n- **Official Language:** Bengali (Bangla).  \n- **Currency:** Bangladeshi Taka (BDT).  \n- **National Motto:** “Victory of the People”.\n\n---\n\n### II. Geography & Environment  \n1. **Physical Setting**  \n   - Predominantly low‑lying delta of the Ganges‑Brahmaputra‑Meghna river system.  \n   - Average elevation ≈\u202f12\u202fm above sea level; extensive floodplains and wetlands.  \n2. **Key Natural Features**  \n   - **Sundarbans:** UNESCO World Heritage mangrove forest, home to the Bengal tiger.  \n   - **Hill Tracts:** Chittagong Hill Tracts region – forested, tribal communities.  \n   - **Coastline:** ~710\u202fkm with numerous rivers, creeks, and 